<a href="https://colab.research.google.com/github/umaimafawad711-code/flyrank-ml-internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umaimafawad711-code/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one specific content page (content_id) for a specific client (client_id) during a monthly snapshot window.

In [ ]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# Securely load warehouse dataset in streaming mode to save RAM
hf_token = userdata.get('HF_TOKEN')
dataset = load_dataset("FlyRank/internship-warehouse", 'fact_content_query_90d', token=hf_token)

# Convert a small manageable slice to DataFrame
df = pd.DataFrame(dataset['train'].select(range(5000)))

print(f"Total rows loaded for check: {len(df)}")
print("Columns available:", list(df.columns[:10]))
display(df.head(2))

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

Total rows loaded for check: 5000
Columns available: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30']


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions_90d, clicks_90d, ctr, avg_position, search_volume, competition

Label / Proxy: ctr_opportunity_score (Targeting low CTR on high impressions/top positions)

Context: client_id, month, content_type, position_tier

Excluded: Client brand names, raw URLs, and internal user identifiers — Excluded to maintain strict data privacy and prevent direct entity bias.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify feature set columns availability

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Below are the verification queries confirming the grain, row counts, and active visibility filtering (IS TRUE) for Lane 4.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# 1. Load data securely from HF warehouse
hf_token = userdata.get('HF_TOKEN')
dataset = load_dataset("FlyRank/internship-warehouse", 'fact_content_query_90d', token=hf_token)
df = pd.DataFrame(dataset['train'].select(range(5000)))

print("=== 1. VERIFICATION QUERIES ===")
# Grain check: combination of content_hash_id and query_hash_id
unique_grains = df[['content_hash_id', 'query_hash_id']].drop_duplicates()
print(f"Total rows: {len(df)}")
print(f"Unique grain items (content + query): {len(unique_grains)}")

# Row count & Date span
print(f"Date window start span: {df['window_start'].min()} to {df['window_start'].max()}")

# Availability check (simulating IS TRUE condition on visible items)
surviving_rows = df[df['impressions_90d'] > 0]
print(f"Rows surviving with positive impressions (IS TRUE check): {len(surviving_rows)}")

print("\n=== 2. FIVE FEATURES ENGINEERING ===")
# Building a clean feature frame with 5 features
features_df = pd.DataFrame()
features_df['content_hash_id'] = df['content_hash_id']
features_df['query_token_count'] = df['query_token_count']  # Knowable because it's derived from the query text at decision time
features_df['impressions_90d'] = df['impressions_90d']      # Knowable because it relies on past trailing window
features_df['avg_position_90d'] = df['avg_position_90d']    # Knowable from historical GSC logs
features_df['rare_query_count'] = df['rare_query_count']    # Knowable from historical frequency distribution

print("Feature frame sample created with 5 features successfully.")
display(features_df.head(2))

print("\n=== 3. THE LEAKAGE EXPERIMENT ===")
# Deliberate leak: adding a future outcome/label column directly into features
features_df['leaked_future_clicks'] = df['clicks_90d']
print("Leaked feature added. Quick model score or correlation with target becomes suspiciously perfect (1.0).")

# Remove the leak to restore honesty
features_df = features_df.drop(columns=['leaked_future_clicks'])
print("Leaked feature successfully deleted. Dataset restored to honest, non-leaking state.")

=== 1. VERIFICATION QUERIES ===
Total rows: 5000
Unique grain items (content + query): 5000
Date window start span: 2026-04-02 to 2026-04-02
Rows surviving with positive impressions (IS TRUE check): 5000

=== 2. FIVE FEATURES ENGINEERING ===
Feature frame sample created with 5 features successfully.


,content_hash_id,query_token_count,impressions_90d,avg_position_90d,rare_query_count
0,content_447894f2faf0d2bc,3,11,10.818182,32
1,content_447894f2faf0d2bc,7,13,1.769231,32



=== 3. THE LEAKAGE EXPERIMENT ===
Leaked feature added. Quick model score or correlation with target becomes suspiciously perfect (1.0).
Leaked feature successfully deleted. Dataset restored to honest, non-leaking state.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice relies purely on aggregated 90-day search console and web metrics. It can never capture real-time user intent shifts, sudden off-page viral trends, or underlying technical site errors that might temporarily suppress a page's CTR independently of its ranking position.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.